# Camada Silver — `ecommerce_produtos` 

Lê a Bronze física em `az://squad1/bronze/ecommerce_produtos`, aplica as 10 regras de qualidade, grava **somente linhas válidas** na Silver física em `az://squad1/silver/ecommerce_produtos` e registra as falhas em `az://squad1/dq_monitoring_logs`.

Este notebook possui modo de reprocessamento para quando os arquivos já foram lidos anteriormente.

In [0]:
%run ../utils/utils

##  Imports e parâmetros

In [0]:

#  Carrega as funções utilitárias (gravar_delta, ler_delta, etc)

import uuid
import pyspark.sql.functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window
from datetime import datetime, timezone
from functools import reduce

# Variáveis do Processo
RUN_ID = str(uuid.uuid4())
TABELA_ALVO = "ecommerce_produtos"
TABELA_DQ = "dq_monitoring_logs"
DATA_EXECUCAO = datetime.now(timezone.utc)

print(f"Iniciando processamento Silver - Produtos - Run ID: {RUN_ID}")

## Leitura do Micro-lote e Tabelas de Referência (Joins)

In [0]:
# 1. Carrega a tabela Bronze de Produtos
try:
    df_bronze_produtos = ler_delta("bronze", TABELA_ALVO, STORAGE_OPTIONS)
except Exception as e:
    raise Exception(f"Erro: A tabela Bronze de {TABELA_ALVO} não foi encontrada.")

if delta_existe("silver", TABELA_ALVO, STORAGE_OPTIONS):
    df_silver_atual = ler_delta("silver", TABELA_ALVO, STORAGE_OPTIONS)

    if delta_existe("silver/quarentena", TABELA_ALVO, STORAGE_OPTIONS):
        df_quarentena = ler_delta("silver/quarentena", TABELA_ALVO, STORAGE_OPTIONS)
        df_processados = df_silver_atual.select("sku").union(df_quarentena.select("sku"))
    else:
        df_processados = df_silver_atual.select("sku")

    df_micro_lote = df_bronze_produtos.join(df_processados, "sku", "left_anti")
else:
    df_micro_lote = df_bronze_produtos

qtd_novos = df_micro_lote.count()
print(f"Registros novos no micro-lote de produtos para processar: {qtd_novos}")

# Referência de Categorias (Para Regras 4 e 10)
df_categorias_ref = obter_referencia_silver_ou_bronze("ecommerce_categorias", ["id_categoria", "id_categoria_pai"]) \
    .withColumnRenamed("id_categoria", "id_categoria_ref") \
    .withColumnRenamed("id_categoria_pai", "id_categoria_pai_ref")

if df_categorias_ref.count() == 0:
    print("[AVISO CRÍTICO] df_categorias_ref está VAZIO. A R4 (categoria FK) vai reprovar TODOS os "
          "produtos até a Silver de ecommerce_categorias ter pelo menos uma linha válida.")

# =================================================================================
# Referência de Vendas dos últimos 90 dias (Para Regra 9)
# CORREÇÃO: agora prioriza a Silver de itens_pedido e cai para a Bronze como
# fallback. Como a Bronze não tem a coluna "silver_processed_at" (só existe
# depois que o dado passa pela Silver), usamos "bronze_ingested_at" como
# proxy de data de referência quando caímos no fallback.
# =================================================================================
if delta_existe("silver", "ecommerce_itens_pedido", STORAGE_OPTIONS):
    print("Referência de ecommerce_itens_pedido (R9 produtos): usando Silver.")
    df_itens_ref = ler_delta("silver", "ecommerce_itens_pedido", STORAGE_OPTIONS)
    coluna_data_referencia = "silver_processed_at"
elif delta_existe("bronze", "ecommerce_itens_pedido", STORAGE_OPTIONS):
    print("Referência de ecommerce_itens_pedido (R9 produtos): Silver ainda não existe — usando Bronze.")
    df_itens_ref = ler_delta("bronze", "ecommerce_itens_pedido", STORAGE_OPTIONS)
    coluna_data_referencia = "bronze_ingested_at"
else:
    print("Referência de ecommerce_itens_pedido (R9 produtos): nem Silver nem Bronze encontradas.")
    df_itens_ref = None
    coluna_data_referencia = None

if df_itens_ref is not None:
    df_vendas_90d = (
        df_itens_ref
        .filter(F.datediff(F.current_date(), F.col(coluna_data_referencia)) <= 90)
        .select("sku").dropDuplicates()
        .withColumn("vendido_90d", F.lit(True))
    )
else:
    schema_vendas = StructType([StructField("sku", StringType(), True), StructField("vendido_90d", BooleanType(), True)])
    df_vendas_90d = spark.createDataFrame([], schema_vendas)

print("Tabelas de referência cruzada carregadas com sucesso.")

In [0]:
print("Total de linhas na Bronze:", df_bronze_produtos.count())
print("SKUs distintos na Bronze:", df_bronze_produtos.select("sku").distinct().count())

## Leitura da Bronze e Aplicação das 10 Regras de Data Quality



In [0]:
if qtd_novos > 0:
    # Domínios permitidos (Regra 5)
    unidades_validas = ['un', 'kg', 'g', 'L', 'ml']

    # Normalização de is_ativo
    valores_ativo_true = ["true", "1", "s", "sim", "y", "yes", "verdadeiro", "ativo"]
    valores_ativo_false = ["false", "0", "n", "nao", "não", "no", "falso", "inativo"]
    is_ativo_norm = F.trim(F.lower(F.col("is_ativo").cast("string")))

    # Window com orderBy explícito por bronze_ingested_at: garante que "o
    # primeiro encontrado" é determinístico entre execuções. row_number() == 1
    # marca o sobrevivente; qualquer linha com row_number() > 1 é duplicata.
    w_sku = Window.partitionBy("sku").orderBy(F.col("bronze_ingested_at").asc())

    # Prepara o DataFrame aplicando os acoplamentos e Joins
    df_base = (df_micro_lote
        .withColumn("preco_lista_num", F.col("preco_lista").cast("double"))
        .withColumn("id_categoria_num", F.col("id_categoria").cast("long"))
        .withColumn(
            "is_ativo_bool",
            F.when(is_ativo_norm.isin(valores_ativo_true), F.lit(True))
             .when(is_ativo_norm.isin(valores_ativo_false), F.lit(False))
             .otherwise(F.lit(None))
        )
        .withColumn("unidade_medida_norm", F.trim(F.col("unidade_medida")))
        .withColumn("row_sku_no_grupo", F.row_number().over(w_sku))
        .join(df_categorias_ref, F.col("id_categoria").cast("long") == df_categorias_ref.id_categoria_ref, "left")
        .join(df_vendas_90d, "sku", "left"))

    # Tratamento para nulos da tabela de vendas
    df_base = df_base.fillna({"vendido_90d": False})

    # Aplicação massiva das 10 regras unificadas (Todas tratadas como impeditivas)
    df_silver_produtos = (df_base
        .withColumn(
            "r1_sku_falhou",
            # Nulo/vazio sempre reprova; duplicata só reprova a partir da
            # SEGUNDA ocorrência (row_sku_no_grupo > 1). A primeira ocorrência
            # (row_sku_no_grupo == 1) é validada.
            F.col("sku").isNull() | (F.trim(F.col("sku")) == "") | (F.col("row_sku_no_grupo") > 1)
        )
        .withColumn("r2_preco_lista_falhou", F.col("preco_lista_num").isNull() | (F.col("preco_lista_num") <= 0))
        .withColumn("r3_nome_produto_falhou", F.col("nome_produto").isNull() | (F.trim(F.col("nome_produto")) == ""))
        .withColumn("r4_id_categoria_fk_falhou", F.col("id_categoria_num").isNull() | F.col("id_categoria_ref").isNull())
        .withColumn("r5_unidade_medida_falhou", F.col("unidade_medida_norm").isNull() | (~F.col("unidade_medida_norm").isin(unidades_validas)))
        .withColumn("r6_is_ativo_falhou", F.col("is_ativo_bool").isNull())
        .withColumn("r7_preco_teto_falhou", F.col("preco_lista_num").isNotNull() & (F.col("preco_lista_num") > 5000.0))
        .withColumn("r8_nome_marca_falhou", F.col("nome_marca").isNull() | (F.trim(F.col("nome_marca")) == ""))
        .withColumn("r9_produto_ativo_sem_venda_falhou", (F.col("is_ativo_bool") == True) & (F.col("vendido_90d") == False))
        .withColumn("r10_categoria_raiz_falhou", F.col("id_categoria_ref").isNotNull() & F.col("id_categoria_pai_ref").isNull()))

    # =================================================================================
    # 📌 SISTEMA DE DIAGNÓSTICO ATUALIZADO
    # =================================================================================
    print(f"--- DETALHAMENTO DE FALHAS NO LOTE DE PRODUTOS ---")
    print(f"Total de registros analisados no micro-lote: {df_silver_produtos.count()}")

    regras_verificacao = [
        ("R1 (SKU Nulo/Duplicado)", "r1_sku_falhou"),
        ("R2 (Preço Lista Inválido)", "r2_preco_lista_falhou"),
        ("R3 (Nome Produto Vazio)", "r3_nome_produto_falhou"),
        ("R4 (Categoria FK Inexistente)", "r4_id_categoria_fk_falhou"),
        ("R5 (Unidade Medida Fora Padrão)", "r5_unidade_medida_falhou"),
        ("R6 (is_ativo Nulo/Inválido)", "r6_is_ativo_falhou"),
        ("R7 (Preço Excede Teto)", "r7_preco_teto_falhou"),
        ("R8 (Marca Obrigatória Vazia)", "r8_nome_marca_falhou"),
        ("R9 (Produto Ativo Sem Giro 90D)", "r9_produto_ativo_sem_venda_falhou"),
        ("R10 (Produto em Categoria Raiz)", "r10_categoria_raiz_falhou")
    ]

    for nome_regra, nome_coluna in regras_verificacao:
        falhas = df_silver_produtos.filter(F.col(nome_coluna) == True).count()
        print(f"⚠️ {nome_regra}: Encontrou {falhas} linhas com erro.")
    print(f"---------------------------------------------------")

    # Estruturação Estratégica do Catálogo para o dq_monitoring_logs
    catalogo_regras = [
        {"coluna": "r1_sku_falhou", "regra": "R1_SKU_NULO_DUPLICADO", "severidade": "Critica"},
        {"coluna": "r2_preco_lista_falhou", "regra": "R2_PRECO_LISTA_INVALIDO", "severidade": "Critica"},
        {"coluna": "r3_nome_produto_falhou", "regra": "R3_NOME_PRODUTO_VAZIO", "severidade": "Critica"},
        {"coluna": "r4_id_categoria_fk_falhou", "regra": "R4_CATEGORIA_FK_INEXISTENTE", "severidade": "Critica"},
        {"coluna": "r5_unidade_medida_falhou", "regra": "R5_UNIDADE_MEDIDA_FORA_PADRAO", "severidade": "Critica"},
        {"coluna": "r6_is_ativo_falhou", "regra": "R6_IS_ATIVO_NULO_INVALIDO", "severidade": "Critica"},
        {"coluna": "r7_preco_teto_falhou", "regra": "R7_PRECO_EXCEDE_TETO_SECOS", "severidade": "Critica"},
        {"coluna": "r8_nome_marca_falhou", "regra": "R8_MARCA_OBRIGATORIA_VAZIA", "severidade": "Critica"},
        {"coluna": "r9_produto_ativo_sem_venda_falhou", "regra": "R9_PROD_ATIVO_SEM_GIRO_90D", "severidade": "Critica"},
        {"coluna": "r10_categoria_raiz_falhou", "regra": "R10_PRODUTO_EM_CATEGORIA_RAIZ", "severidade": "Critica"}
    ]

    total_registros = df_silver_produtos.count()
    logs_list = []

    for r in catalogo_regras:
        qtd_falhas = df_silver_produtos.filter(F.col(r["coluna"]) == True).count()
        if qtd_falhas > 0:
            logs_list.append((
                RUN_ID, TABELA_ALVO, r["regra"], "FAIL", r["severidade"],
                int(qtd_falhas), int(total_registros), datetime.now(timezone.utc), f"Bronze Delta ({TABELA_ALVO})"
            ))

    if logs_list:
        df_dq_monitoring_logs_novos = spark.createDataFrame(logs_list, schema_dq_logs())
    else:
        df_dq_monitoring_logs_novos = spark.createDataFrame([], schema_dq_logs())

    # --- CRITÉRIO UNIFICADO DE SELEÇÃO ---
    todas_flags = [r["coluna"] for r in catalogo_regras]
    condicao_total_falha = reduce(lambda a, b: a | b, [F.col(c) for c in todas_flags])

    df_silver_produtos = (df_silver_produtos
        .withColumn("silver_linha_valida", ~condicao_total_falha)
        .withColumn("silver_processed_at", F.current_timestamp())
        .withColumn("silver_run_id", F.lit(RUN_ID)))

    print("Muralha de qualidade unificada para produtos estruturada com sucesso.")
else:
    df_dq_monitoring_logs_novos = spark.createDataFrame([], schema_dq_logs())
    print("Etapa ignorada: não há micro-lote novo.")


## Gravação Final (Silver Clientes e Logs)

In [0]:
# =================================================================================
# 3. GRAVAÇÃO BLINDADA E ADERÊNCIA AO NOVO CONTRATO ASYNC
# =================================================================================
if qtd_novos > 0:
    colunas_finais = df_micro_lote.columns + ["silver_processed_at", "silver_run_id"]

    # A. Gravação dos Aprovados (Silver)
    df_silver_validos = (df_silver_produtos
        .filter(F.col("silver_linha_valida") == True)
        .select(*colunas_finais))

    qtd_validos = df_silver_validos.count()
    print(f"Produtos aprovados para a Silver: {qtd_validos}")

    if qtd_validos > 0:
        gravar_delta(
            df=df_silver_validos, camada="silver", tabela=TABELA_ALVO,
            storage_opts=STORAGE_OPTIONS, mode="append", particionar=False
        )

    # B. Gravação dos Reprovados (Quarentena com Anti-Join por SKU)
    df_silver_invalidos = (df_silver_produtos
        .filter(F.col("silver_linha_valida") == False)
        .select(*colunas_finais))

    if df_silver_invalidos.count() > 0:
        if delta_existe("silver/quarentena", TABELA_ALVO, STORAGE_OPTIONS):
            df_quarentena_historico = ler_delta("silver/quarentena", TABELA_ALVO, STORAGE_OPTIONS)
            df_quarentena_para_gravar = df_silver_invalidos.join(
                df_quarentena_historico.select("sku"),
                on="sku",
                how="left_anti"
            )
        else:
            df_quarentena_para_gravar = df_silver_invalidos

        qtd_novos_rejeitados = df_quarentena_para_gravar.count()
        if qtd_novos_rejeitados > 0:
            gravar_delta(
                df=df_quarentena_para_gravar, camada="silver/quarentena", tabela=TABELA_ALVO,
                storage_opts=STORAGE_OPTIONS, mode="append", particionar=False
            )
            print(f"Enviados {qtd_novos_rejeitados} produtos novos para a quarentena.")

    # C. Gravação dos Logs de Auditoria
    if df_dq_monitoring_logs_novos.count() > 0:
        gravar_delta(
            df=df_dq_monitoring_logs_novos, camada="", tabela=TABELA_DQ,
            storage_opts=STORAGE_OPTIONS, mode="append", particionar=False
        )
        print("Logs de qualidade consolidados na raiz!")
else:
    print("Rotina finalizada sem alterações físicas.")


##  Validação Final

In [0]:
print("===== VALIDAÇÃO FINAL =====")

if delta_existe("silver", TABELA_ALVO, STORAGE_OPTIONS):
    df_validacao_silver = ler_delta("silver", TABELA_ALVO, STORAGE_OPTIONS)
    print(f"Registros totais na Silver {TABELA_ALVO}:", df_validacao_silver.count())
    display(df_validacao_silver.limit(20))
else:
    print(f"Aviso: tabela Silver {TABELA_ALVO} não encontrada.")

# CORREÇÃO: dq_monitoring_logs fica na RAIZ do container (camada=""), não em
# "silver/". A versão anterior checava o caminho certo no "if" mas lia do
# caminho errado ("silver") dentro do bloco — os dois precisam usar camada="".
if delta_existe("", "dq_monitoring_logs", STORAGE_OPTIONS):
    df_logs_validacao = ler_delta("", "dq_monitoring_logs", STORAGE_OPTIONS) \
        .filter(F.col("tabela") == TABELA_ALVO)
    print(f"Total de violações registradas para {TABELA_ALVO}:", df_logs_validacao.count())
    display(df_logs_validacao.orderBy(F.col("timestamp_execucao").desc()).limit(20))
else:
    print("Aviso: tabela dq_monitoring_logs não encontrada na raiz do Data Lake.")
